# Attention variables & final dataset
Merges Google Trends, Instagram and club data onto `performance.csv` for final dataset `players_cross_section.csv`.

In [2]:
import os
BASE = os.path.abspath('dissertation_data')
os.makedirs(BASE, exist_ok=True)

In [3]:
# Configuration
PERF_CSV      = os.path.join(BASE, 'performance.csv')
IG_TEMPLATE   = os.path.join(BASE, 'instagram_club_manual.csv')
TRENDS_MANUAL = os.path.join(BASE, 'google_trends_manual.csv')
OUT_CROSS     = os.path.join(BASE, 'players_cross_section.csv')
TRENDS_TIMEFRAME = '2024-08-01 2025-07-13'   # season start to Club World Cup final

In [4]:
# Helpers
import re, unicodedata
import pandas as pd, numpy as np
def norm_name(name):
    if not isinstance(name, str): return ''
    s = unicodedata.normalize('NFKD', name).encode('ascii','ignore').decode()
    s = re.sub(r'[^a-zA-Z ]', '', s).strip().lower()
    return re.sub(r'\s+', ' ', s)
def classify_position(raw):
    if not isinstance(raw, str) or not raw.strip(): return None
    s = re.split(r'[,/]', raw.strip().lower())[0].strip()
    if s=='gk' or 'goalkeep' in s or 'keeper' in s: return 'Goalkeeper'
    if 'midfield' in s or s in ('mf','cm','dm','am','cdm','cam'): return 'Midfielder'
    if 'back' in s or 'defen' in s or s=='df' or s in ('cb','rb','lb','rwb','lwb'): return 'Defender'
    if ('forward' in s or 'wing' in s or 'strik' in s or 'attack' in s or s=='fw'
            or s in ('cf','ss','rw','lw','st')): return 'Forward'
    return None

In [5]:
assert os.path.exists(PERF_CSV), 'performance.csv not found'
data = pd.read_csv(PERF_CSV)
print('Loaded performance:', len(data), 'players')

Loaded performance: 146 players


## Google Trends (collected manually):
1. `google_trends_links.csv` has one link per player which compares the player with a fixed
   reference term (Cristiano Ronaldo) over the season window to puts all players on one scale.
2. download from each link and put the files in `trends_csv/`.
3. then run the parser to builds `google_trends_monthly.csv` and `google_trends_manual.csv` —
   then re-run the merge cell.

In [13]:
# Trend Fetch/Merge Google Trends averages (from the parser below)
if os.path.exists(TRENDS_MANUAL):
    m = pd.read_csv(TRENDS_MANUAL); m['name_key'] = m['player_name'].map(norm_name)
    data = data.drop(columns=[c for c in data.columns if c.startswith('google_trends')], errors='ignore')
    data = data.merge(m[['name_key','google_trends']], on='name_key', how='left')
    print('google_trends filled for', int(data['google_trends'].notna().sum()), '/', len(data))
else:
    import urllib.parse
    path = os.path.join(BASE, 'google_trends_links.csv')
    if not os.path.exists(path):
        rows = [{'player_name': p,
                 'trends_link': 'https://trends.google.com/trends/explore?date=' +
                 urllib.parse.quote(TRENDS_TIMEFRAME) + '&q=' +
                 urllib.parse.quote(f'Cristiano Ronaldo,{p}')}
                for p in data['player_name'] if p != 'Cristiano Ronaldo']
        pd.DataFrame(rows).to_csv(path, index=False)
    print('No google_trends_manual.csv yet — collect via', path, 'then run the parser below.')

google_trends filled for 146 / 146


In [11]:
# after uploading a zip of downloads
import glob, shutil
tdir = os.path.join(BASE, 'trends_csv'); os.makedirs(tdir, exist_ok=True)
for z in glob.glob(os.path.join(tdir, '*.zip')):
    shutil.unpack_archive(z, tdir)
shutil.rmtree(os.path.join(tdir, '__MACOSX'), ignore_errors=True)
for f in glob.glob(os.path.join(tdir, '**', '*.csv'), recursive=True):
    if os.path.dirname(f) != tdir: shutil.move(f, tdir)
print('CSV files in trends_csv/:', len(glob.glob(os.path.join(tdir, '*.csv'))))

CSV files in trends_csv/: 145


In [12]:
# Parse the downloaded comparison CSVs -> monthly series + per-player averages.
import pandas as pd, numpy as np, os, glob, re, unicodedata, difflib

TRENDS_DIR = os.path.join(BASE, 'trends_csv')
os.makedirs(TRENDS_DIR, exist_ok=True)
ANCHOR = 'Cristiano Ronaldo'

SPECIAL = str.maketrans({'ı':'i','İ':'I','ø':'o','Ø':'O','đ':'d','Đ':'D','ł':'l','Ł':'L',
                         'ß':'ss','æ':'ae','Æ':'AE','œ':'oe','Œ':'OE'})
def _norm(name):
    if not isinstance(name, str): return ''
    s = name.translate(SPECIAL)
    s = unicodedata.normalize('NFKD', s).encode('ascii','ignore').decode()
    s = re.sub(r'[^a-zA-Z ]','',s).strip().lower()
    return re.sub(r'\s+',' ',s)

# canonical names = the dataset's players (so outputs merge cleanly)
canonical = data['player_name'].dropna().unique().tolist()
_keys = {_norm(n): n for n in canonical}
def match_player(term):
    k = _norm(term)
    if k in _keys: return _keys[k]                                  # 1 exact (accent-insensitive)
    kt = set(k.split())
    subset = [n for nk,n in _keys.items() if kt <= set(nk.split()) or set(nk.split()) <= kt]
    if len(subset) == 1: return subset[0]                           # 2 short vs full form
    sur = k.split()[-1] if k else ''
    bysur = [n for nk,n in _keys.items() if nk.split()[-1] == sur]
    if len(bysur) == 1: return bysur[0]                             # 3 unique surname (safe)
    close = difflib.get_close_matches(k, list(_keys), n=1, cutoff=0.85)
    if close: return _keys[close[0]]                                # 4 fuzzy, last + stricter
    return None

files = sorted(glob.glob(os.path.join(TRENDS_DIR, '*.csv')))
print(f'Found {len(files)} downloaded Trends CSVs in {TRENDS_DIR}')

def parse_trends_csv(path):
    raw = open(path, encoding='utf-8-sig').read().splitlines()
    hdr = next((i for i,l in enumerate(raw) if l.lower().startswith(('week','month','day'))), None)
    if hdr is None: return None
    d = pd.read_csv(path, skiprows=hdr)
    cols = list(d.columns)
    terms = {c: re.sub(r':\s*\(.*\)$', '', c).strip() for c in cols[1:]}
    anchor_col = next((c for c,t in terms.items() if _norm(t) == _norm(ANCHOR)), None)
    player_col = next((c for c,t in terms.items() if _norm(t) != _norm(ANCHOR)), None)
    if anchor_col is None or player_col is None: return None
    for c in [anchor_col, player_col]:
        d[c] = pd.to_numeric(d[c].astype(str).str.replace('<1','0.5',regex=False), errors='coerce')
    d = d.rename(columns={cols[0]:'date', player_col:'player_score', anchor_col:'anchor_score'})
    d['date'] = pd.to_datetime(d['date'], errors='coerce')
    return terms[player_col], d[['date','player_score','anchor_score']]

monthly_rows, avg_rows, skipped, unmatched = [], [], [], []
for f in files:
    parsed = parse_trends_csv(f)
    if parsed is None: skipped.append(os.path.basename(f)); continue
    raw_term, d = parsed
    pname = match_player(raw_term)
    if pname is None:
        unmatched.append(f'{os.path.basename(f)} (term: {raw_term!r})'); continue
    anchor_mean = d['anchor_score'].mean()
    if not anchor_mean or np.isnan(anchor_mean) or anchor_mean == 0:
        skipped.append(os.path.basename(f)); continue
    d['rel'] = d['player_score'] / anchor_mean
    m = d.set_index('date')['rel'].resample('MS').mean().round(4)
    for dt, v in m.items():
        monthly_rows.append({'player_name': pname, 'month': dt.strftime('%Y-%m'), 'trends_rel': v})
    avg_rows.append({'player_name': pname, 'rel': d['rel'].mean(),
                     'floor_share': float((d['player_score'] <= 0.5).mean())})

if skipped:   print('Skipped (unreadable / bad anchor):', skipped[:8])
if unmatched: print('UNMATCHED terms (tell Claude these):', unmatched[:8])

if avg_rows:
    pd.DataFrame(monthly_rows).to_csv(os.path.join(BASE, 'google_trends_monthly.csv'), index=False)
    res = pd.DataFrame(avg_rows).drop_duplicates('player_name')
    floored = res.loc[res['floor_share'] > 0.5, 'player_name'].tolist()
    if floored:
        print(f'Note: {len(floored)} players sat at the "<1" floor most weeks (fine, but noted):',
              ', '.join(floored[:6]), '...' if len(floored) > 6 else '')
    top = res['rel'].max()
    res['google_trends'] = (res['rel'] / max(top, 1.0) * 100).round(2)
    anchor_gt = round(100 / max(top, 1.0), 2) if top > 1 else 100.0
    out = pd.concat([res[['player_name','google_trends']],
                     pd.DataFrame([{'player_name': ANCHOR, 'google_trends': anchor_gt}])],
                    ignore_index=True)
    out.to_csv(os.path.join(BASE, 'google_trends_manual.csv'), index=False)
    done = set(out['player_name'])
    left = [p for p in canonical if p not in done]
    print(f'Parsed & matched {len(res)} players | missing: {len(left)}')
    if left: print('  ', ', '.join(left[:10]))
    print('Wrote google_trends_monthly.csv and google_trends_manual.csv.')
    print('Re-run the Trends fetch cell above to merge.')
else:
    print('No usable CSVs yet')


Found 145 downloaded Trends CSVs in /content/dissertation_data/trends_csv
Note: 12 players sat at the "<1" floor most weeks (fine, but noted): André-Frank Zambo Anguissa, Tino Livramento, Mikel Oyarzabal, Álvaro Carreras, Joan García, Daniel Muñoz ...
Parsed & matched 145 players | missing: 0
Wrote google_trends_monthly.csv and google_trends_manual.csv.
Re-run the Trends fetch cell above to merge.


## Instagram + two-season club details (manual)
Player Instagram is a single mid-2026 snapshot. Club, league and club Instagram following are recorded
for **both** seasons so transfers are captured.

In [17]:
if not os.path.exists(IG_TEMPLATE):
    t = data[['player_name']].copy()
    t['club_2024_25']=data.get('club',''); t['league_2024_25']=data.get('league','')
    t['ig_followers']=''; t['engagement_rate']=''
    t['club_2025_26']=''; t['league_2025_26']=''
    t['club_ig_followers_2024_25']=''; t['club_ig_followers_2025_26']=''
    t.to_csv(IG_TEMPLATE, index=False)
    print('Wrote', IG_TEMPLATE, '- fill it, then re-run.')
ig = pd.read_csv(IG_TEMPLATE); ig['name_key']=ig['player_name'].map(norm_name)
for col in ['ig_followers','engagement_rate','club_ig_followers_2024_25','club_ig_followers_2025_26']:
    if col in ig.columns: ig[col]=pd.to_numeric(ig[col].astype(str).str.replace(',',''),errors='coerce')

stale = ['ig_followers','engagement_rate','club_2025_26','league_2025_26',
         'club_ig_followers_2024_25','club_ig_followers_2025_26',
         'club_ig_followers','transferred','moved_up']
data = data.drop(columns=[c for c in stale if c in data.columns], errors='ignore')

keep=['name_key','ig_followers','engagement_rate','club_2025_26','league_2025_26',
      'club_ig_followers_2024_25','club_ig_followers_2025_26']
data = data.merge(ig[[c for c in keep if c in ig.columns]], on='name_key', how='left')
data['club_2024_25']=data.get('club',''); data['league_2024_25']=data.get('league','')
nk=lambda x: norm_name(x) if isinstance(x,str) else ''
moved = data['club_2025_26'].map(nk).ne('') & (data['club_2024_25'].map(nk)!=data['club_2025_26'].map(nk))
data['transferred']=moved.astype('Int64')
data['moved_up']=(data['club_ig_followers_2025_26']>data['club_ig_followers_2024_25']).astype('Int64')
data['club_ig_followers']=data['club_ig_followers_2024_25']
print('Instagram filled for', int(data['ig_followers'].notna().sum()),'/',len(data),
      '| club IG filled:', int(data['club_ig_followers_2024_25'].notna().sum()),
      '| transferred:', int(data['transferred'].fillna(0).sum()))
data.head()

Instagram filled for 144 / 146 | club IG filled: 146 | transferred: 33


,player_name,name_key,in_ballondor,consensus_count,club,league,position,position_group,nationality,club_season_start,...,league_2024_25,ig_followers,engagement_rate,club_2025_26,league_2025_26,club_ig_followers_2024_25,club_ig_followers_2025_26,transferred,moved_up,club_ig_followers
0,Ousmane Dembélé,ousmane dembele,True,3,Paris Saint-Germain,Ligue 1,Forward,Forward,France,Paris Saint-Germain,...,Ligue 1,22175754.0,8.30,Paris Saint-Germain,Ligue 1,66864756,66864756,0,0,66864756
1,Lamine Yamal,lamine yamal,True,3,Barcelona,La Liga,Forward,Forward,Spain,Barcelona,...,La Liga,45992501.0,8.13,Barcelona,La Liga,145936297,145936297,0,0,145936297
2,Vitinha,vitinha,True,3,Paris Saint-Germain,Ligue 1,Midfielder,Midfielder,Portugal,Paris Saint-Germain,...,Ligue 1,5235860.0,6.94,Paris Saint-Germain,Ligue 1,66864756,66864756,0,0,66864756
3,Mohamed Salah,mohamed salah,True,3,Liverpool,Premier League,Forward,Forward,Egypt,Liverpool,...,Premier League,66754774.0,1.98,Liverpool,Premier League,48467485,48467485,0,0,48467485
4,Raphinha,raphinha,True,3,Barcelona,La Liga,Forward,Forward,Brazil,Barcelona,...,La Liga,20697158.0,4.43,Barcelona,La Liga,145936297,145936297,0,0,145936297


## Build the final dataset

In [18]:
data['player_id']=[f'P{i:03d}' for i in range(len(data))]
cross_cols=[
    # profile
    'player_id','player_name','nationality','position_group',
    'club_2024_25','league_2024_25','club_season_start','mid_season_tranfers',
    'club_2025_26','league_2025_26','transferred','moved_up',
    # attention (outcomes)
    'ig_followers','engagement_rate','google_trends',
    # performance
    'appearances','minutes','goals','assists','G_plus_A','goals_p90','assists_p90',
    'shots_pg','dribbles','fouled','offsides','dispossessed',
    'key_passes','passes_pg','pass_pct','crosses','long_balls',
    'tackles','interceptions','fouls','clearances','blocks','Whoscored_Rating',
    # goalkeepers
    'gk_goals_against','gk_save_pct','gk_clean_sheets',
    # team context & brand
    'team_league_position','ucl_stage','ucl_stage_ord',
    'club_ig_followers','club_ig_followers_2024_25','club_ig_followers_2025_26',
    # sample-frame
    'in_ballondor','consensus_count']
cross = data[[c for c in cross_cols if c in data.columns]]
cross.to_csv(OUT_CROSS, index=False)

print('Wrote', OUT_CROSS)
try:
    from google.colab import files; files.download(OUT_CROSS)
except Exception: pass
cross.head()

Wrote /content/dissertation_data/players_cross_section.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,player_id,player_name,nationality,position_group,club_2024_25,league_2024_25,club_season_start,mid_season_tranfers,club_2025_26,league_2025_26,...,gk_save_pct,gk_clean_sheets,team_league_position,ucl_stage,ucl_stage_ord,club_ig_followers,club_ig_followers_2024_25,club_ig_followers_2025_26,in_ballondor,consensus_count
0,P000,Ousmane Dembélé,France,Forward,Paris Saint-Germain,Ligue 1,Paris Saint-Germain,0,Paris Saint-Germain,Ligue 1,...,NaN,NaN,1,Winner,7,66864756,66864756,66864756,True,3
1,P001,Lamine Yamal,Spain,Forward,Barcelona,La Liga,Barcelona,0,Barcelona,La Liga,...,NaN,NaN,1,Semi Final,5,145936297,145936297,145936297,True,3
2,P002,Vitinha,Portugal,Midfielder,Paris Saint-Germain,Ligue 1,Paris Saint-Germain,0,Paris Saint-Germain,Ligue 1,...,NaN,NaN,1,Winner,7,66864756,66864756,66864756,True,3
3,P003,Mohamed Salah,Egypt,Forward,Liverpool,Premier League,Liverpool,0,Liverpool,Premier League,...,NaN,NaN,1,Round of 16,3,48467485,48467485,48467485,True,3
4,P004,Raphinha,Brazil,Forward,Barcelona,La Liga,Barcelona,0,Barcelona,La Liga,...,NaN,NaN,1,Semi Final,5,145936297,145936297,145936297,True,3
